In [ ]:
# Re-define implant and model if needed
import pulse2percept as p2p
implant = p2p.implants.ArgusII()
model_p2p = p2p.models.ScoreboardModel()
model_p2p.build()

def encode_image_to_stimulus(image_gray, implant, amplitude=150):
    n = len(implant.electrode_names)
    cols = int(np.ceil(np.sqrt(n * 1.5)))
    rows = int(np.ceil(n / cols))
    clahe = cv2.createCLAHE(clipLimit=6.0, tileGridSize=(4,4))
    enhanced = clahe.apply(image_gray)
    blurred = cv2.GaussianBlur(enhanced, (5,5), 0)
    edges = cv2.Canny(blurred, 10, 40)
    combined = 0.35 * enhanced + 0.65 * edges
    resized = cv2.resize(combined, (cols, rows))
    mn, mx = resized.min(), resized.max()
    stretched = (resized - mn) / (mx - mn + 1e-8)
    amplitudes_flat = stretched.flatten()[:n]
    stimulus_data = {}
    for i, name in enumerate(implant.electrode_names):
        stimulus_data[name] = amplitudes_flat[i] * amplitude
    return p2p.stimuli.Stimulus(stimulus_data)

def compute_ssim(percept_data, img_original):
    from skimage.metrics import structural_similarity as ssim
    h, w = percept_data.shape
    original_resized = cv2.resize(img_original, (w, h)) / 255.0
    perceived_norm = (percept_data - percept_data.min()) / (percept_data.max() - percept_data.min() + 1e-8)
    return ssim(perceived_norm, original_resized, data_range=1.0)

In [3]:
import pulse2percept as p2p
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# ─────────────────────────────────────────────
# FUNCTION: Smart Saliency Mask
# ─────────────────────────────────────────────

def get_saliency_mask_v2(img_gray):
    h, w = img_gray.shape
    mean_brightness = img_gray.mean()
    print(f"Mean brightness: {mean_brightness:.1f}")

    if mean_brightness > 180:
        print("→ White background detected, using threshold method")

        blurred = cv2.GaussianBlur(img_gray, (5, 5), 0)
        _, thresh = cv2.threshold(
            blurred, 0, 255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        kernel = np.ones((5, 5), np.uint8)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

        contours, _ = cv2.findContours(
            thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        if not contours:
            print("No contours found, using full image")
            return np.ones((h, w), dtype=np.uint8), thresh, thresh, (0, 0, w, h)

        largest = max(contours, key=cv2.contourArea)
        clean_mask = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(clean_mask, [largest], -1, 1, thickness=cv2.FILLED)
        x, y, bw, bh = cv2.boundingRect(largest)
        return clean_mask, thresh, thresh, (x, y, bw, bh)

    else:
        print("→ Complex background detected, using GrabCut")

        img_bgr = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)
        margin = 0.15
        rx = int(w * margin)
        ry = int(h * margin)
        rw = int(w * (1 - 2 * margin))
        rh = int(h * (1 - 2 * margin))

        # Make sure rect is valid
        if rw < 10 or rh < 10:
            print("Image too small for GrabCut")
            return np.ones((h, w), dtype=np.uint8), \
                   np.ones((h, w), dtype=np.uint8) * 255, \
                   np.ones((h, w), dtype=np.uint8) * 255, \
                   (0, 0, w, h)

        mask = np.zeros((h, w), np.uint8)
        bgd = np.zeros((1, 65), np.float64)
        fgd = np.zeros((1, 65), np.float64)

        try:
            cv2.grabCut(
                img_bgr, mask, (rx, ry, rw, rh),
                bgd, fgd, 5, cv2.GC_INIT_WITH_RECT
            )
            clean_mask = np.where(
                (mask == 2) | (mask == 0), 0, 1
            ).astype(np.uint8)
            print(f"GrabCut success ✓")

        except Exception as e:
            print(f"GrabCut failed: {e}")
            # Fallback: adaptive threshold
            blurred = cv2.GaussianBlur(img_gray, (11, 11), 0)
            adaptive = cv2.adaptiveThreshold(
                blurred, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY_INV, 21, 5
            )
            kernel = np.ones((7, 7), np.uint8)
            adaptive = cv2.morphologyEx(adaptive, cv2.MORPH_CLOSE, kernel)
            contours, _ = cv2.findContours(
                adaptive, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            clean_mask = np.zeros((h, w), dtype=np.uint8)
            if contours:
                largest = max(contours, key=cv2.contourArea)
                cv2.drawContours(
                    clean_mask, [largest], -1, 1, thickness=cv2.FILLED
                )

        thresh_display = (clean_mask * 255).astype(np.uint8)
        contours_f, _ = cv2.findContours(
            thresh_display, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if contours_f:
            x, y, bw, bh = cv2.boundingRect(
                max(contours_f, key=cv2.contourArea)
            )
        else:
            x, y, bw, bh = 0, 0, w, h

        return clean_mask, thresh_display, thresh_display, (x, y, bw, bh)


# ─────────────────────────────────────────────
# FUNCTION: Encode with Saliency
# ─────────────────────────────────────────────

def encode_with_saliency_v2(img_gray, implant, amplitude=150):
    n = len(implant.electrode_names)
    cols = int(np.ceil(np.sqrt(n * 1.5)))
    rows = int(np.ceil(n / cols))

    clean_mask, _, _, _ = get_saliency_mask_v2(img_gray)

    # Apply mask
    focused = img_gray * clean_mask

    # Multi-scale edges
    b1 = cv2.GaussianBlur(focused, (3, 3), 0)
    b2 = cv2.GaussianBlur(focused, (7, 7), 0)
    b3 = cv2.GaussianBlur(focused, (15, 15), 0)

    e1 = cv2.Canny(b1, 50, 150).astype(np.float32) / 255.0
    e2 = cv2.Canny(b2, 30, 90).astype(np.float32) / 255.0
    e3 = cv2.Canny(b3, 10, 40).astype(np.float32) / 255.0

    edges = 0.5 * e1 + 0.3 * e2 + 0.2 * e3
    brightness = focused.astype(np.float32) / 255.0
    combined = 0.3 * brightness + 0.7 * edges

    mn, mx = combined.min(), combined.max()
    stretched = (combined - mn) / (mx - mn + 1e-8)

    resized = cv2.resize(stretched, (cols, rows))
    amplitudes_flat = resized.flatten()[:n]

    stimulus_data = {}
    for i, name in enumerate(implant.electrode_names):
        stimulus_data[name] = amplitudes_flat[i] * amplitude

    return p2p.stimuli.Stimulus(stimulus_data)


# ─────────────────────────────────────────────
# LOAD IMAGE — change path here
# ─────────────────────────────────────────────

# List all files in your folder first
folder = "/Users/rangoju/Desktop/train_images_v2/"
print("Files in folder:")
for f in sorted(os.listdir(folder)):
    print(" ", f)

# Change this to one of the filenames printed above
IMG_PATH = folder + "sample.jpg"

img = cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)

if img is None:
    from PIL import Image
    img = np.array(Image.open(IMG_PATH).convert("L"))
    print("Loaded via PIL")

print(f"Image loaded: {img.shape}")


# ─────────────────────────────────────────────
# RUN SALIENCY PIPELINE
# ─────────────────────────────────────────────

clean_mask, thresh, _, bbox = get_saliency_mask_v2(img)
foreground_only = img * clean_mask

pct = 100 * clean_mask.sum() / clean_mask.size
print(f"Foreground: {clean_mask.sum()} / {clean_mask.size} pixels ({pct:.1f}%)")

# Show pipeline
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title("1. Original")
axes[0].axis('off')

axes[1].imshow(thresh, cmap='gray')
axes[1].set_title("2. Threshold / GrabCut")
axes[1].axis('off')

axes[2].imshow(clean_mask, cmap='gray')
axes[2].set_title("3. Object Mask")
axes[2].axis('off')

axes[3].imshow(foreground_only, cmap='gray')
axes[3].set_title("4. Object Only")
axes[3].axis('off')

plt.suptitle("Saliency Pipeline v2", fontsize=13)
plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────
# COMPARE OLD vs NEW ENCODER
# ─────────────────────────────────────────────

# Old encoder
stimulus_old = encode_image_to_stimulus(img, implant)
implant.stim = stimulus_old
percept_old = model_p2p.predict_percept(implant)

# New encoder
stimulus_new = encode_with_saliency_v2(img, implant)
implant.stim = stimulus_new
percept_new = model_p2p.predict_percept(implant)

def normalize(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

old_score = compute_ssim(percept_old.data[:,:,0], img)
new_score = compute_ssim(percept_new.data[:,:,0], img)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img, cmap='gray')
axes[0].set_title("Camera Input")
axes[0].axis('off')

axes[1].imshow(normalize(percept_old.data[:,:,0]), cmap='gray')
axes[1].set_title(f"Old Encoder\nSSIM: {old_score:.4f}")
axes[1].axis('off')

axes[2].imshow(normalize(percept_new.data[:,:,0]), cmap='gray')
axes[2].set_title(f"New Saliency Encoder\nSSIM: {new_score:.4f}")
axes[2].axis('off')

plt.suptitle("Encoder Comparison — Old vs New", fontsize=14)
plt.tight_layout()
plt.show()

Files in folder:
  -.png
  3.jpg
  Figure_1.png
  WhatsApp Image 2026-05-30 at 9.35.01 PM (2).jpeg
  WhatsApp_Image_2026-05-30_at_10.54.00_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.54.15_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.54.47_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.04_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.18_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.33_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.56.35_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.56.51_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.57.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.58.21_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.59.34_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM. (1).jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM.-1.jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.00.41_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.01.15_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.02.02_PM..jpg
  WhatsApp_Image_2026-05-30_at_

: 

In [7]:
import pulse2percept as p2p
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from skimage.metrics import structural_similarity as ssim

# ─────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────

implant = p2p.implants.ArgusII()
n_electrodes = len(implant.electrode_names)
model_p2p = p2p.models.ScoreboardModel()
model_p2p.build()
print(f"Implant: ArgusII | Electrodes: {n_electrodes}")
print("Model ready ✓")

# ─────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────

def normalize(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

def compute_ssim(percept_data, img_original):
    h, w = percept_data.shape
    original_resized = cv2.resize(img_original, (w, h)) / 255.0
    perceived_norm = normalize(percept_data)
    return ssim(perceived_norm, original_resized, data_range=1.0)

# ─────────────────────────────────────────────
# OLD ENCODER
# ─────────────────────────────────────────────

def encode_image_to_stimulus(image_gray, implant, amplitude=150):
    n = len(implant.electrode_names)
    cols = int(np.ceil(np.sqrt(n * 1.5)))
    rows = int(np.ceil(n / cols))

    clahe = cv2.createCLAHE(clipLimit=6.0, tileGridSize=(4, 4))
    enhanced = clahe.apply(image_gray)
    blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)
    edges_tight = cv2.Canny(blurred, 10, 40)
    edges_loose = cv2.Canny(blurred, 5, 20)
    edges = cv2.addWeighted(edges_tight, 0.6, edges_loose, 0.4, 0)
    combined = 0.35 * enhanced + 0.65 * edges
    resized = cv2.resize(combined, (cols, rows))
    mn, mx = resized.min(), resized.max()
    stretched = (resized - mn) / (mx - mn + 1e-8)
    amplitudes_flat = stretched.flatten()[:n]

    stimulus_data = {}
    for i, name in enumerate(implant.electrode_names):
        stimulus_data[name] = amplitudes_flat[i] * amplitude

    return p2p.stimuli.Stimulus(stimulus_data)

# ─────────────────────────────────────────────
# SALIENCY MASK
# ─────────────────────────────────────────────

def get_saliency_mask_v2(img_gray):
    h, w = img_gray.shape
    mean_brightness = img_gray.mean()
    print(f"  Mean brightness: {mean_brightness:.1f}")

    if mean_brightness > 180:
        print("  → White background: using threshold")

        blurred = cv2.GaussianBlur(img_gray, (5, 5), 0)
        _, thresh = cv2.threshold(
            blurred, 0, 255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )
        kernel = np.ones((5, 5), np.uint8)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

        contours, _ = cv2.findContours(
            thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            return np.ones((h, w), dtype=np.uint8), thresh, thresh, (0, 0, w, h)

        largest = max(contours, key=cv2.contourArea)
        clean_mask = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(clean_mask, [largest], -1, 1, thickness=cv2.FILLED)
        x, y, bw, bh = cv2.boundingRect(largest)
        return clean_mask, thresh, thresh, (x, y, bw, bh)

    else:
        print("  → Complex background: using GrabCut")

        img_bgr = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)
        margin = 0.15
        rx = int(w * margin)
        ry = int(h * margin)
        rw = int(w * (1 - 2 * margin))
        rh = int(h * (1 - 2 * margin))

        mask = np.zeros((h, w), np.uint8)
        bgd = np.zeros((1, 65), np.float64)
        fgd = np.zeros((1, 65), np.float64)

        try:
            cv2.grabCut(
                img_bgr, mask, (rx, ry, rw, rh),
                bgd, fgd, 5, cv2.GC_INIT_WITH_RECT
            )
            clean_mask = np.where(
                (mask == 2) | (mask == 0), 0, 1
            ).astype(np.uint8)
            print("  GrabCut success ✓")

        except Exception as e:
            print(f"  GrabCut failed: {e} → using adaptive threshold")
            blurred = cv2.GaussianBlur(img_gray, (11, 11), 0)
            adaptive = cv2.adaptiveThreshold(
                blurred, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY_INV, 21, 5
            )
            kernel = np.ones((7, 7), np.uint8)
            adaptive = cv2.morphologyEx(adaptive, cv2.MORPH_CLOSE, kernel)
            contours, _ = cv2.findContours(
                adaptive, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            clean_mask = np.zeros((h, w), dtype=np.uint8)
            if contours:
                largest = max(contours, key=cv2.contourArea)
                cv2.drawContours(
                    clean_mask, [largest], -1, 1, thickness=cv2.FILLED
                )

        thresh_display = (clean_mask * 255).astype(np.uint8)
        contours_f, _ = cv2.findContours(
            thresh_display, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if contours_f:
            x, y, bw, bh = cv2.boundingRect(
                max(contours_f, key=cv2.contourArea)
            )
        else:
            x, y, bw, bh = 0, 0, w, h

        return clean_mask, thresh_display, thresh_display, (x, y, bw, bh)

# ─────────────────────────────────────────────
# NEW ENCODER
# ─────────────────────────────────────────────

def encode_with_saliency_v2(image_gray, implant, amplitude=150):
    n = len(implant.electrode_names)
    cols = int(np.ceil(np.sqrt(n * 1.5)))
    rows = int(np.ceil(n / cols))

    # Get foreground mask
    clean_mask, _, _, _ = get_saliency_mask_v2(image_gray)

    # Apply mask — zero out background
    focused = image_gray * clean_mask

    # Multi-scale edge detection
    b1 = cv2.GaussianBlur(focused, (3, 3), 0)
    b2 = cv2.GaussianBlur(focused, (7, 7), 0)
    b3 = cv2.GaussianBlur(focused, (15, 15), 0)

    e1 = cv2.Canny(b1, 50, 150).astype(np.float32) / 255.0
    e2 = cv2.Canny(b2, 30, 90).astype(np.float32) / 255.0
    e3 = cv2.Canny(b3, 10, 40).astype(np.float32) / 255.0

    edges = 0.5 * e1 + 0.3 * e2 + 0.2 * e3
    brightness = focused.astype(np.float32) / 255.0
    combined = 0.3 * brightness + 0.7 * edges

    mn, mx = combined.min(), combined.max()
    stretched = (combined - mn) / (mx - mn + 1e-8)

    resized = cv2.resize(stretched, (cols, rows))
    amplitudes_flat = resized.flatten()[:n]

    stimulus_data = {}
    for i, name in enumerate(implant.electrode_names):
        stimulus_data[name] = amplitudes_flat[i] * amplitude

    return p2p.stimuli.Stimulus(stimulus_data)

# ─────────────────────────────────────────────
# LOAD IMAGE
# ─────────────────────────────────────────────

folder = "/Users/rangoju/Desktop/Retinal-Prosthesis/train_images_v2/"
print("\nFiles available:")
for f in sorted(os.listdir(folder)):
    print(f"  {f}")

# ← CHANGE THIS to any filename printed above
IMG_PATH = folder + "3.jpg"

img = cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)
if img is None:
    from PIL import Image as PILImage
    img = np.array(PILImage.open(IMG_PATH).convert("L"))
    print("Loaded via PIL")

print(f"\nImage loaded: {img.shape}")

# ─────────────────────────────────────────────
# SHOW SALIENCY PIPELINE
# ─────────────────────────────────────────────

print("\nRunning saliency pipeline...")
clean_mask, thresh, _, bbox = get_saliency_mask_v2(img)
foreground_only = img * clean_mask
pct = 100 * clean_mask.sum() / clean_mask.size
print(f"  Foreground: {pct:.1f}% of image")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title("1. Original")
axes[0].axis('off')
axes[1].imshow(thresh, cmap='gray')
axes[1].set_title("2. Threshold")
axes[1].axis('off')
axes[2].imshow(clean_mask, cmap='gray')
axes[2].set_title("3. Object Mask")
axes[2].axis('off')
axes[3].imshow(foreground_only, cmap='gray')
axes[3].set_title("4. Object Only")
axes[3].axis('off')
plt.suptitle("Saliency Pipeline", fontsize=13)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────
# RUN BOTH ENCODERS
# ─────────────────────────────────────────────

print("\nRunning old encoder...")
stimulus_old = encode_image_to_stimulus(img, implant)
implant.stim = stimulus_old
percept_old = model_p2p.predict_percept(implant)

print("Running new saliency encoder...")
stimulus_new = encode_with_saliency_v2(img, implant)
implant.stim = stimulus_new
percept_new = model_p2p.predict_percept(implant)

old_score = compute_ssim(percept_old.data[:, :, 0], img)
new_score = compute_ssim(percept_new.data[:, :, 0], img)

print(f"\nOld encoder SSIM: {old_score:.4f}")
print(f"New encoder SSIM: {new_score:.4f}")
improvement = ((new_score - old_score) / (abs(old_score) + 1e-8)) * 100
print(f"Improvement:      {improvement:+.1f}%")

# ─────────────────────────────────────────────
# FINAL COMPARISON PLOT
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img, cmap='gray')
axes[0].set_title("Camera Input", fontsize=13)
axes[0].axis('off')

axes[1].imshow(normalize(percept_old.data[:, :, 0]), cmap='gray')
axes[1].set_title(f"Old Encoder\nSSIM: {old_score:.4f}", fontsize=13)
axes[1].axis('off')

axes[2].imshow(normalize(percept_new.data[:, :, 0]), cmap='gray')
axes[2].set_title(f"New Saliency Encoder\nSSIM: {new_score:.4f}", fontsize=13)
axes[2].axis('off')

plt.suptitle("Encoder Comparison — Old vs New", fontsize=15)
plt.tight_layout()
plt.show()

Implant: ArgusII | Electrodes: 60
Model ready ✓

Files available:
  .DS_Store
  3.jpg
  WhatsApp Image 2026-05-30 at 9.35.01 PM (2).jpeg
  WhatsApp_Image_2026-05-30_at_10.57.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.58.21_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.59.34_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM.-1.jpg
  WhatsApp_Image_2026-05-30_at_9.34.56_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.56_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.34.57_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.57_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.34.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.58_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.34.58_PM_2..jpg
  WhatsApp_Image_2026-05-30_at_9.34.59_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.59_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.35.00_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.35.00_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.35.01_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.35.01_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.35.02_PM..j

Files:
  -.png
  3.jpg
  Figure_1.png
  WhatsApp Image 2026-05-30 at 9.35.01 PM (2).jpeg
  WhatsApp_Image_2026-05-30_at_10.54.00_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.54.15_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.54.47_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.04_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.18_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.33_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.55.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.56.35_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.56.51_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.57.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.58.21_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.59.34_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM. (1).jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM.-1.jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.00.41_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.01.15_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.02.02_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.02.18_P

In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def extract_foreground_v2(img_color):
    h, w = img_color.shape[:2]
    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

    # Step 1: Mean shift smooth
    smoothed = img_color.copy()
    cv2.pyrMeanShiftFiltering(img_color, 2, 10, smoothed, 4)
    hsv = cv2.cvtColor(smoothed, cv2.COLOR_BGR2HSV)

    # Step 2: Build BACKGROUND histogram from border pixels only
    # Border = definitely background, not the object
    border_mask = np.zeros((h, w), dtype=np.uint8)
    border = 20  # pixel thickness of border strip
    border_mask[:border, :] = 255        # top
    border_mask[h-border:, :] = 255      # bottom
    border_mask[:, :border] = 255        # left
    border_mask[:, w-border:] = 255      # right

    # Calculate histogram of BACKGROUND only
    bg_hist = cv2.calcHist(
        [hsv], [0, 1], border_mask,
        [16, 16], [0, 180, 0, 256]
    )
    cv2.normalize(bg_hist, bg_hist, 0, 255, cv2.NORM_MINMAX)

    # Step 3: Backproject — pixels similar to background get HIGH value
    backproj = cv2.calcBackProject(
        [hsv], [0, 1], bg_hist,
        [0, 180, 0, 256], 1
    )

    # Smooth
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    cv2.filter2D(backproj, -1, kernel, backproj)

    # Step 4: Invert — foreground becomes bright
    foreground_prob = cv2.bitwise_not(backproj)
    cv2.normalize(foreground_prob, foreground_prob, 0, 255, cv2.NORM_MINMAX)

    # Threshold
    _, thresh = cv2.threshold(
        foreground_prob, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Clean up
    kernel_clean = np.ones((15, 15), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_clean)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)

    # Step 5: Largest contour = main object
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        print("No contours — returning full image")
        return np.ones((h, w), dtype=np.uint8), thresh, foreground_prob, img_gray

    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    x, y, bw, bh = cv2.boundingRect(contours[0])
    pad = 20
    x  = max(0, x - pad)
    y  = max(0, y - pad)
    bw = min(w - x, bw + 2 * pad)
    bh = min(h - y, bh + 2 * pad)

    # Step 6: GrabCut refinement
    mask = np.zeros((h, w), np.uint8)
    bgd  = np.zeros((1, 65), np.float64)
    fgd  = np.zeros((1, 65), np.float64)

    try:
        cv2.grabCut(
            img_color, mask, (x, y, bw, bh),
            bgd, fgd, 5, cv2.GC_INIT_WITH_RECT
        )
        clean_mask = np.where(
            (mask == 2) | (mask == 0), 0, 1
        ).astype(np.uint8)
        print("GrabCut ✓")

    except Exception as e:
        print(f"GrabCut failed: {e}")
        clean_mask = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(clean_mask, [contours[0]], -1, 1, cv2.FILLED)

    foreground = img_gray * clean_mask
    return clean_mask, thresh, foreground_prob, foreground


# ─────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────

folder = "/Users/rangoju/Desktop/Retinal-Prosthesis/train_images_v2/"
print("Files:")
for f in sorted(os.listdir(folder)):
    print(f"  {f}")

IMG_PATH = folder + "3.jpg"  # ← change to your banana file

img_color = cv2.imread(IMG_PATH)
if img_color is None:
    raise FileNotFoundError(f"Could not load: {IMG_PATH}")

img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
print(f"\nLoaded: {img_color.shape}")

# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────

clean_mask, thresh, backproj, foreground = extract_foreground_v2(img_color)
pct = 100 * clean_mask.sum() / clean_mask.size
print(f"Foreground: {pct:.1f}%")

# ─────────────────────────────────────────────
# SHOW
# ─────────────────────────────────────────────

img_bbox = img_color.copy()
contours_f, _ = cv2.findContours(
    (clean_mask * 255).astype(np.uint8),
    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)
if contours_f:
    x, y, bw, bh = cv2.boundingRect(max(contours_f, key=cv2.contourArea))
    cv2.rectangle(img_bbox, (x, y), (x+bw, y+bh), (0, 255, 0), 3)

fig, axes = plt.subplots(1, 6, figsize=(28, 5))

axes[0].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
axes[0].set_title("1. Original")
axes[0].axis('off')

axes[1].imshow(backproj, cmap='hot')
axes[1].set_title("2. Foreground Probability\nbright = object")
axes[1].axis('off')

axes[2].imshow(thresh, cmap='gray')
axes[2].set_title("3. Important Regions")
axes[2].axis('off')

axes[3].imshow(clean_mask, cmap='gray')
axes[3].set_title("4. GrabCut Mask")
axes[3].axis('off')

axes[4].imshow(foreground, cmap='gray')
axes[4].set_title("5. Clean Foreground")
axes[4].axis('off')

axes[5].imshow(cv2.cvtColor(img_bbox, cv2.COLOR_BGR2RGB))
axes[5].set_title("6. Detected Object")
axes[5].axis('off')

plt.suptitle("Foreground Extraction v2 — Border-based Background", fontsize=14)
plt.tight_layout()
plt.show()

Files:
  .DS_Store
  3.jpg
  WhatsApp Image 2026-05-30 at 9.35.01 PM (2).jpeg
  WhatsApp_Image_2026-05-30_at_10.57.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.58.21_PM..jpg
  WhatsApp_Image_2026-05-30_at_10.59.34_PM..jpg
  WhatsApp_Image_2026-05-30_at_11.00.07_PM.-1.jpg
  WhatsApp_Image_2026-05-30_at_9.34.56_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.56_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.34.57_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.57_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.34.58_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.58_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.34.58_PM_2..jpg
  WhatsApp_Image_2026-05-30_at_9.34.59_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.34.59_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.35.00_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.35.00_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.35.01_PM..jpg
  WhatsApp_Image_2026-05-30_at_9.35.01_PM_1..jpg
  WhatsApp_Image_2026-05-30_at_9.35.02_PM..jpg
  sample.jpg
  test.jpg
  test1.jpg
  testimg.jpeg
  tes

In [ ]:
import pulse2percept as p2p
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────

implant = p2p.implants.ArgusII()
model_p2p = p2p.models.ScoreboardModel()
model_p2p.build()
print("Ready ✓")

# ─────────────────────────────────────────────
# STEP 1: Understand electrode layout
# ─────────────────────────────────────────────

# Print all electrode positions so we understand the grid
print("\nArgusII electrode positions (microns from fovea):")
xs, ys, names = [], [], []
for name, elec in implant.electrodes.items():
    xs.append(elec.x)
    ys.append(elec.y)
    names.append(name)
    
xs = np.array(xs)
ys = np.array(ys)

print(f"X range: {xs.min():.0f} to {xs.max():.0f} microns")
print(f"Y range: {ys.min():.0f} to {ys.max():.0f} microns")
print(f"Total electrodes: {len(names)}")

# Visualize electrode grid
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(xs, ys, c='yellow', s=200, edgecolors='white', zorder=5)
for i, name in enumerate(names):
    ax.annotate(name, (xs[i], ys[i]), 
                textcoords="offset points", xytext=(0,8),
                color='white', fontsize=7, ha='center')
ax.set_facecolor('black')
ax.set_xlabel('x (microns from fovea)')
ax.set_ylabel('y (microns from fovea)')
ax.set_title('ArgusII — 60 Electrode Positions on Retina')
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────
# STEP 2: Foreground extraction (from previous code)
# ─────────────────────────────────────────────

def extract_foreground_v2(img_color):
    h, w = img_color.shape[:2]
    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
    smoothed = img_color.copy()
    cv2.pyrMeanShiftFiltering(img_color, 2, 10, smoothed, 4)
    hsv = cv2.cvtColor(smoothed, cv2.COLOR_BGR2HSV)
    
    # Border-based background histogram
    border_mask = np.zeros((h, w), dtype=np.uint8)
    b = 20
    border_mask[:b,:] = 255
    border_mask[h-b:,:] = 255
    border_mask[:,:b] = 255
    border_mask[:,w-b:] = 255
    
    bg_hist = cv2.calcHist([hsv], [0,1], border_mask, [16,16], [0,180,0,256])
    cv2.normalize(bg_hist, bg_hist, 0, 255, cv2.NORM_MINMAX)
    
    backproj = cv2.calcBackProject([hsv], [0,1], bg_hist, [0,180,0,256], 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11,11))
    cv2.filter2D(backproj, -1, kernel, backproj)
    
    fg_prob = cv2.bitwise_not(backproj)
    cv2.normalize(fg_prob, fg_prob, 0, 255, cv2.NORM_MINMAX)
    equalized = cv2.equalizeHist(fg_prob)
    
    _, thresh = cv2.threshold(equalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel_clean = np.ones((15,15), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_clean)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.ones((h,w), dtype=np.uint8), img_gray
    
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    x, y, bw, bh = cv2.boundingRect(contours[0])
    pad = 20
    x = max(0,x-pad); y = max(0,y-pad)
    bw = min(w-x,bw+2*pad); bh = min(h-y,bh+2*pad)
    
    mask = np.zeros((h,w), np.uint8)
    bgd = np.zeros((1,65), np.float64)
    fgd = np.zeros((1,65), np.float64)
    
    try:
        cv2.grabCut(img_color, mask, (x,y,bw,bh), bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
        clean_mask = np.where((mask==2)|(mask==0), 0, 1).astype(np.uint8)
    except:
        clean_mask = np.zeros((h,w), dtype=np.uint8)
        cv2.drawContours(clean_mask, [contours[0]], -1, 1, cv2.FILLED)
    
    foreground = img_gray * clean_mask
    return clean_mask, foreground

# ─────────────────────────────────────────────
# STEP 3: The correct encoder
# Maps image → real ArgusII electrode positions
# ─────────────────────────────────────────────

def encode_to_argusII(img_color, implant, amplitude=150):
    """
    Correct encoding:
    1. Extract foreground
    2. Build feature map (edges + brightness)
    3. For each electrode, sample image at that
       electrode's ACTUAL retinal position
    4. Return stimulus
    """
    h, w = img_color.shape[:2]
    
    # Step A: Extract foreground
    clean_mask, foreground = extract_foreground_v2(img_color)
    print(f"  Foreground: {100*clean_mask.sum()/clean_mask.size:.1f}%")
    
    # Step B: Build feature map on foreground
    fg = foreground.copy()
    
    # Multi-scale edges
    b1 = cv2.GaussianBlur(fg, (3,3), 0)
    b2 = cv2.GaussianBlur(fg, (7,7), 0)
    b3 = cv2.GaussianBlur(fg, (15,15), 0)
    
    e1 = cv2.Canny(b1, 30, 90).astype(np.float32) / 255.0
    e2 = cv2.Canny(b2, 20, 60).astype(np.float32) / 255.0
    e3 = cv2.Canny(b3, 10, 40).astype(np.float32) / 255.0
    
    edges = 0.5*e1 + 0.3*e2 + 0.2*e3
    brightness = fg.astype(np.float32) / 255.0
    
    # Blend: edges matter more for recognition
    feature_map = 0.3 * brightness + 0.7 * edges
    
    # Stretch to full 0-1 range
    mn, mx = feature_map.min(), feature_map.max()
    feature_map = (feature_map - mn) / (mx - mn + 1e-8)
    
    # Step C: Get real electrode positions
    xs, ys, elec_names = [], [], []
    for name, elec in implant.electrodes.items():
        xs.append(elec.x)
        ys.append(elec.y)
        elec_names.append(name)
    
    xs = np.array(xs, dtype=np.float32)
    ys = np.array(ys, dtype=np.float32)
    
    # Normalize electrode positions to image pixel coordinates
    # ArgusII x spans left→right, y spans bottom→top
    # Image pixel x spans left→right, y spans top→bottom
    # So we flip y
    xs_norm = (xs - xs.min()) / (xs.max() - xs.min() + 1e-8)
    ys_norm = (ys - ys.min()) / (ys.max() - ys.min() + 1e-8)
    ys_norm = 1.0 - ys_norm  # flip y axis to match image coordinates
    
    # Step D: Sample feature map at each electrode position
    amplitudes = np.zeros(len(elec_names))
    radius = 18  # sampling neighborhood in pixels
    
    for i in range(len(elec_names)):
        # Map normalized position to image pixel
        px = int(xs_norm[i] * (w - 1))
        py = int(ys_norm[i] * (h - 1))
        
        # Sample neighborhood
        x1 = max(0, px - radius)
        x2 = min(w, px + radius)
        y1 = max(0, py - radius)
        y2 = min(h, py + radius)
        
        patch = feature_map[y1:y2, x1:x2]
        amplitudes[i] = patch.mean() if patch.size > 0 else 0.0
    
    # Final normalization
    if amplitudes.max() > 0:
        amplitudes = amplitudes / amplitudes.max()
    
    # Step E: Build stimulus dict
    stimulus_data = {}
    for i, name in enumerate(elec_names):
        stimulus_data[name] = amplitudes[i] * amplitude
    
    return p2p.stimuli.Stimulus(stimulus_data), amplitudes, elec_names

# ─────────────────────────────────────────────
# STEP 4: Run on your image
# ─────────────────────────────────────────────

IMG_PATH = "/Users/rangoju/Desktop/Retina-Prosthesis/train_images_v2/3.jpg"  # ← change this
img_color = cv2.imread(IMG_PATH)
img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
print(f"Image: {img_color.shape}")

print("\nEncoding...")
stimulus, amplitudes, elec_names = encode_to_argusII(img_color, implant)
implant.stim = stimulus

print("Simulating perception...")
percept = model_p2p.predict_percept(implant)

# ─────────────────────────────────────────────
# STEP 5: Visualize everything
# ─────────────────────────────────────────────

def normalize(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

# Show electrode activation pattern as grid
# ArgusII is 6 rows x 10 cols
amp_grid = amplitudes.reshape(6, 10)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# 1. Original
axes[0].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
axes[0].set_title("1. Camera Input", fontsize=12)
axes[0].axis('off')

# 2. Foreground
_, foreground = extract_foreground_v2(img_color)
axes[1].imshow(foreground, cmap='gray')
axes[1].set_title("2. Extracted Foreground", fontsize=12)
axes[1].axis('off')

# 3. Electrode activation pattern — this is what gets sent to implant
im = axes[2].imshow(amp_grid, cmap='hot', aspect='auto')
axes[2].set_title("3. Electrode Activation Pattern\n(6×10 ArgusII grid)", fontsize=12)
axes[2].set_xlabel("Electrode column")
axes[2].set_ylabel("Electrode row")
plt.colorbar(im, ax=axes[2], label='Amplitude (normalized)')

# Add electrode names
for row in range(6):
    for col in range(10):
        idx = row * 10 + col
        if idx < len(elec_names):
            axes[2].text(col, row, elec_names[idx],
                        ha='center', va='center',
                        color='white', fontsize=6)

# 4. Perceived output
axes[3].imshow(normalize(percept.data[:,:,0]), cmap='gray')
axes[3].set_title("4. Patient Perception\n(pulse2percept simulation)", fontsize=12)
axes[3].axis('off')

plt.suptitle("Full Pipeline: Image → Foreground → Electrodes → Perception", fontsize=13)
plt.tight_layout()
plt.show()

# Print stats
from skimage.metrics import structural_similarity as ssim_fn
h_p, w_p = percept.data[:,:,0].shape
orig_resized = cv2.resize(img_gray, (w_p, h_p)) / 255.0
perc_norm = normalize(percept.data[:,:,0])
score = ssim_fn(perc_norm, orig_resized, data_range=1.0)
print(f"\nSSIM Score: {score:.4f}")
print(f"Active electrodes: {(amplitudes > 0.1).sum()} / {len(amplitudes)}")
print(f"Max amplitude: {amplitudes.max():.3f}")
print(f"Mean amplitude: {amplitudes.mean():.3f}")

Ready ✓

ArgusII electrode positions (microns from fovea):
X range: -2588 to 2588 microns
Y range: -1438 to 1438 microns
Total electrodes: 60


[ WARN:0@487.347] global loadsave.cpp:278 findDecoder imread_('/Users/rangoju/Desktop/train_images_v2/3.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.13.0) /Users/macmini_16g/GHA-Actions-Opencv-Python/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [1]:
import pulse2percept as p2p
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────

implant = p2p.implants.ArgusII()
model_p2p = p2p.models.ScoreboardModel()
model_p2p.build()
print("Ready ✓")

# ─────────────────────────────────────────────
# STEP 1: Understand electrode layout
# ─────────────────────────────────────────────

# Print all electrode positions so we understand the grid
print("\nArgusII electrode positions (microns from fovea):")
xs, ys, names = [], [], []
for name, elec in implant.electrodes.items():
    xs.append(elec.x)
    ys.append(elec.y)
    names.append(name)
    
xs = np.array(xs)
ys = np.array(ys)

print(f"X range: {xs.min():.0f} to {xs.max():.0f} microns")
print(f"Y range: {ys.min():.0f} to {ys.max():.0f} microns")
print(f"Total electrodes: {len(names)}")

# Visualize electrode grid
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(xs, ys, c='yellow', s=200, edgecolors='white', zorder=5)
for i, name in enumerate(names):
    ax.annotate(name, (xs[i], ys[i]), 
                textcoords="offset points", xytext=(0,8),
                color='white', fontsize=7, ha='center')
ax.set_facecolor('black')
ax.set_xlabel('x (microns from fovea)')
ax.set_ylabel('y (microns from fovea)')
ax.set_title('ArgusII — 60 Electrode Positions on Retina')
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────
# STEP 2: Foreground extraction (from previous code)
# ─────────────────────────────────────────────

def extract_foreground_v2(img_color):
    h, w = img_color.shape[:2]
    img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
    smoothed = img_color.copy()
    cv2.pyrMeanShiftFiltering(img_color, 2, 10, smoothed, 4)
    hsv = cv2.cvtColor(smoothed, cv2.COLOR_BGR2HSV)
    
    # Border-based background histogram
    border_mask = np.zeros((h, w), dtype=np.uint8)
    b = 20
    border_mask[:b,:] = 255
    border_mask[h-b:,:] = 255
    border_mask[:,:b] = 255
    border_mask[:,w-b:] = 255
    
    bg_hist = cv2.calcHist([hsv], [0,1], border_mask, [16,16], [0,180,0,256])
    cv2.normalize(bg_hist, bg_hist, 0, 255, cv2.NORM_MINMAX)
    
    backproj = cv2.calcBackProject([hsv], [0,1], bg_hist, [0,180,0,256], 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11,11))
    cv2.filter2D(backproj, -1, kernel, backproj)
    
    fg_prob = cv2.bitwise_not(backproj)
    cv2.normalize(fg_prob, fg_prob, 0, 255, cv2.NORM_MINMAX)
    equalized = cv2.equalizeHist(fg_prob)
    
    _, thresh = cv2.threshold(equalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel_clean = np.ones((15,15), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_clean)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.ones((h,w), dtype=np.uint8), img_gray
    
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    x, y, bw, bh = cv2.boundingRect(contours[0])
    pad = 20
    x = max(0,x-pad); y = max(0,y-pad)
    bw = min(w-x,bw+2*pad); bh = min(h-y,bh+2*pad)
    
    mask = np.zeros((h,w), np.uint8)
    bgd = np.zeros((1,65), np.float64)
    fgd = np.zeros((1,65), np.float64)
    
    try:
        cv2.grabCut(img_color, mask, (x,y,bw,bh), bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
        clean_mask = np.where((mask==2)|(mask==0), 0, 1).astype(np.uint8)
    except:
        clean_mask = np.zeros((h,w), dtype=np.uint8)
        cv2.drawContours(clean_mask, [contours[0]], -1, 1, cv2.FILLED)
    
    foreground = img_gray * clean_mask
    return clean_mask, foreground

# ─────────────────────────────────────────────
# STEP 3: The correct encoder
# Maps image → real ArgusII electrode positions
# ─────────────────────────────────────────────

def encode_to_argusII(img_color, implant, amplitude=150):
    """
    Correct encoding:
    1. Extract foreground
    2. Build feature map (edges + brightness)
    3. For each electrode, sample image at that
       electrode's ACTUAL retinal position
    4. Return stimulus
    """
    h, w = img_color.shape[:2]
    
    # Step A: Extract foreground
    clean_mask, foreground = extract_foreground_v2(img_color)
    print(f"  Foreground: {100*clean_mask.sum()/clean_mask.size:.1f}%")
    
    # Step B: Build feature map on foreground
    fg = foreground.copy()
    
    # Multi-scale edges
    b1 = cv2.GaussianBlur(fg, (3,3), 0)
    b2 = cv2.GaussianBlur(fg, (7,7), 0)
    b3 = cv2.GaussianBlur(fg, (15,15), 0)
    
    e1 = cv2.Canny(b1, 30, 90).astype(np.float32) / 255.0
    e2 = cv2.Canny(b2, 20, 60).astype(np.float32) / 255.0
    e3 = cv2.Canny(b3, 10, 40).astype(np.float32) / 255.0
    
    edges = 0.5*e1 + 0.3*e2 + 0.2*e3
    brightness = fg.astype(np.float32) / 255.0
    
    # Blend: edges matter more for recognition
    feature_map = 0.3 * brightness + 0.7 * edges
    
    # Stretch to full 0-1 range
    mn, mx = feature_map.min(), feature_map.max()
    feature_map = (feature_map - mn) / (mx - mn + 1e-8)
    
    # Step C: Get real electrode positions
    xs, ys, elec_names = [], [], []
    for name, elec in implant.electrodes.items():
        xs.append(elec.x)
        ys.append(elec.y)
        elec_names.append(name)
    
    xs = np.array(xs, dtype=np.float32)
    ys = np.array(ys, dtype=np.float32)
    
    # Normalize electrode positions to image pixel coordinates
    # ArgusII x spans left→right, y spans bottom→top
    # Image pixel x spans left→right, y spans top→bottom
    # So we flip y
    xs_norm = (xs - xs.min()) / (xs.max() - xs.min() + 1e-8)
    ys_norm = (ys - ys.min()) / (ys.max() - ys.min() + 1e-8)
    ys_norm = 1.0 - ys_norm  # flip y axis to match image coordinates
    
    # Step D: Sample feature map at each electrode position
    amplitudes = np.zeros(len(elec_names))
    radius = 18  # sampling neighborhood in pixels
    
    for i in range(len(elec_names)):
        # Map normalized position to image pixel
        px = int(xs_norm[i] * (w - 1))
        py = int(ys_norm[i] * (h - 1))
        
        # Sample neighborhood
        x1 = max(0, px - radius)
        x2 = min(w, px + radius)
        y1 = max(0, py - radius)
        y2 = min(h, py + radius)
        
        patch = feature_map[y1:y2, x1:x2]
        amplitudes[i] = patch.mean() if patch.size > 0 else 0.0
    
    # Final normalization
    if amplitudes.max() > 0:
        amplitudes = amplitudes / amplitudes.max()
    
    # Step E: Build stimulus dict
    stimulus_data = {}
    for i, name in enumerate(elec_names):
        stimulus_data[name] = amplitudes[i] * amplitude
    
    return p2p.stimuli.Stimulus(stimulus_data), amplitudes, elec_names

# ─────────────────────────────────────────────
# STEP 4: Run on your image
# ─────────────────────────────────────────────

IMG_PATH = "/Users/rangoju/Desktop/Retinal-Prosthesis/train_images_v2/3.jpg"  # ← change this
img_color = cv2.imread(IMG_PATH)
img_gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
print(f"Image: {img_color.shape}")

print("\nEncoding...")
stimulus, amplitudes, elec_names = encode_to_argusII(img_color, implant)
implant.stim = stimulus

print("Simulating perception...")
percept = model_p2p.predict_percept(implant)

# ─────────────────────────────────────────────
# STEP 5: Visualize everything
# ─────────────────────────────────────────────

def normalize(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

# Show electrode activation pattern as grid
# ArgusII is 6 rows x 10 cols
amp_grid = amplitudes.reshape(6, 10)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# 1. Original
axes[0].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
axes[0].set_title("1. Camera Input", fontsize=12)
axes[0].axis('off')

# 2. Foreground
_, foreground = extract_foreground_v2(img_color)
axes[1].imshow(foreground, cmap='gray')
axes[1].set_title("2. Extracted Foreground", fontsize=12)
axes[1].axis('off')

# 3. Electrode activation pattern — this is what gets sent to implant
im = axes[2].imshow(amp_grid, cmap='hot', aspect='auto')
axes[2].set_title("3. Electrode Activation Pattern\n(6×10 ArgusII grid)", fontsize=12)
axes[2].set_xlabel("Electrode column")
axes[2].set_ylabel("Electrode row")
plt.colorbar(im, ax=axes[2], label='Amplitude (normalized)')

# Add electrode names
for row in range(6):
    for col in range(10):
        idx = row * 10 + col
        if idx < len(elec_names):
            axes[2].text(col, row, elec_names[idx],
                        ha='center', va='center',
                        color='white', fontsize=6)

# 4. Perceived output
axes[3].imshow(normalize(percept.data[:,:,0]), cmap='gray')
axes[3].set_title("4. Patient Perception\n(pulse2percept simulation)", fontsize=12)
axes[3].axis('off')

plt.suptitle("Full Pipeline: Image → Foreground → Electrodes → Perception", fontsize=13)
plt.tight_layout()
plt.show()

# Print stats
from skimage.metrics import structural_similarity as ssim_fn
h_p, w_p = percept.data[:,:,0].shape
orig_resized = cv2.resize(img_gray, (w_p, h_p)) / 255.0
perc_norm = normalize(percept.data[:,:,0])
score = ssim_fn(perc_norm, orig_resized, data_range=1.0)
print(f"\nSSIM Score: {score:.4f}")
print(f"Active electrodes: {(amplitudes > 0.1).sum()} / {len(amplitudes)}")
print(f"Max amplitude: {amplitudes.max():.3f}")
print(f"Mean amplitude: {amplitudes.mean():.3f}")

Ready ✓

ArgusII electrode positions (microns from fovea):
X range: -2588 to 2588 microns
Y range: -1438 to 1438 microns
Total electrodes: 60


2026-06-02 23:13:06.508 python[24367:1098320] +[IMKClient subclass]: chose IMKClient_Legacy
2026-06-02 23:13:06.508 python[24367:1098320] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


Image: (528, 856, 3)

Encoding...
  Foreground: 18.5%
Simulating perception...

SSIM Score: 0.0010
Active electrodes: 9 / 60
Max amplitude: 1.000
Mean amplitude: 0.105
